## Forged Document Model Training + Unified Evaluation

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

df = pd.read_csv("dataset_outputs/train.csv")

# 🚀 STRICT FEATURE CLEANING (ANTI-LEAKAGE)
leakage_columns = [
    "Label",
    "Detection_Label",   # 🚨 direct leakage
    "Image_Name",        # 🚨 memorization risk
]

X = df.drop(columns=[col for col in leakage_columns if col in df.columns], errors="ignore")
y = df["Label"].astype(int)

print("[INFO] Final feature columns used:")
print(X.columns.tolist())

# Proper stratified split (anti-memorization baseline)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(n_estimators=350, random_state=42, class_weight="balanced")
model.fit(X_train, y_train)


In [ ]:
# Unified evaluation (ML + AML metrics)
y_pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, proba))
print("FPR:", fp / (fp + tn + 1e-6))


In [ ]:
print("\n===== LEAKAGE CHECK =====")

corr = df.corr(numeric_only=True)["Label"].sort_values(ascending=False)
print(corr.head(10))


In [ ]:
# 🚀 REMOVE DOMINANT FEATURES AND RETRAIN
suspicious_features = [
    "Risk_Score",
    "Risk_Consistency",
    "Field_Completeness",
]

X_reduced = X.drop(columns=[col for col in suspicious_features if col in X.columns], errors="ignore")

model_reduced = RandomForestClassifier(n_estimators=200, random_state=42)
model_reduced.fit(X_train[X_reduced.columns], y_train)

y_pred_reduced = model_reduced.predict(X_test[X_reduced.columns])

print("\n===== ABLATION TEST =====")
print("Original Accuracy :", accuracy_score(y_test, y_pred))
print("Reduced Accuracy  :", accuracy_score(y_test, y_pred_reduced))


In [ ]:
y_test_shuffled = np.random.permutation(y_test)

print("\n===== SHUFFLE TEST =====")
print("Shuffled Accuracy:", accuracy_score(y_test_shuffled, y_pred))


In [ ]:
scores = cross_val_score(model, X, y, cv=5)

print("\n===== CROSS VALIDATION =====")
print("CV Accuracy:", scores.mean())


In [ ]:
importances = model.feature_importances_

df_imp = pd.DataFrame({
    "Feature": X.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\n===== TOP FEATURES =====")
print(df_imp.head(10))
